In [1]:
import json
import os
from pathlib import Path
from pprint import pprint
from typing import Optional, Any
from collections import Counter

import torch
import torch.nn as nn
from torch import Tensor
import numpy as np
import wandb
import evaluate


from datasets import load_from_disk, DatasetDict
import transformers
from transformers import AutoModelForTokenClassification, AutoTokenizer,\
DataCollatorForTokenClassification, TrainingArguments, Trainer, EvalPrediction,\
TrainerCallback
from transformers.models.deberta_v2.tokenization_deberta_v2 import DebertaV2Tokenizer
from transformers.models.deberta_v2.modeling_deberta_v2 import DebertaV2ForTokenClassification
# remove transformers unneeded diagnostic messages
from transformers.utils import logging
logging.set_verbosity_error()

transformers.set_seed(42)
torch.manual_seed(42)

In [2]:
DATA_DIR = Path(os.getcwd()).parent / "data"
DATA_DIR.mkdir(exist_ok=True)

LABEL_INFO_PATH = DATA_DIR/"label_info.json"
DATASET_PATH = DATA_DIR/"cleaned_ai4privacy_300k_pii"

MODELS_DIR = Path(os.getcwd()).parent/"models"
MODELS_DIR.mkdir(exist_ok=True)
dataset: DatasetDict = load_from_disk(DATASET_PATH) # type:ignore
# label info
with open(LABEL_INFO_PATH) as f:
    label_info: dict = json.load(f)
label2id: dict[str, int] = label_info["label2id"]
id2label = label_info["id2label"]

In [3]:
# transformer model path
model_path = "microsoft/deberta-v3-small"
# wandb run
run_name = "small-fully_labeled"
run = wandb.init(project="pii-redaction", name=run_name)

# model checkpoint save dir
model_run_dir = MODELS_DIR/run_name

num_train_samples = len(dataset["train"])
batch_size = 16
num_epochs = 1
total_steps = (num_train_samples // batch_size) * num_epochs 
warmup_steps = int(0.1 * total_steps)

training_args = TrainingArguments(
    output_dir=str(model_run_dir),
    report_to="wandb",
    learning_rate=1e-3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    warmup_steps=warmup_steps,
    bf16=True,
    fp16=False,
    tf32=False,
    max_grad_norm=1.0,
    disable_tqdm=False
)

def update_args():
    # updated training args
    training_args.learning_rate = 2e-5 # lower learning rate
    training_args.num_train_epochs = 10
    # lower batch size now that gradient tree is bigger
    training_args.per_device_train_batch_size = 16
    training_args.gradient_accumulation_steps = 1

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/zelluzy/.netrc.
wandb: Currently logged in as: bengid (bengid-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
tokenizer: DebertaV2Tokenizer = AutoTokenizer.from_pretrained(model_path)

model: DebertaV2ForTokenClassification = AutoModelForTokenClassification.from_pretrained(
    model_path, num_labels=len(id2label), id2label=id2label, label2id=label2id)

assert tokenizer.is_fast

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [5]:
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")

In [6]:
for param in model.base_model.parameters():
    param.requires_grad = False
    
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name} requires grad")

classifier.weight requires grad
classifier.bias requires grad


In [7]:
Counter([param.dtype for param in model.parameters()])

Counter({torch.float16: 104})

In [8]:
for name, module in model.named_modules():
    module.to(torch.float32) # type:ignore

In [9]:
Counter([param.dtype for param in model.parameters()])

Counter({torch.float32: 104})

In [10]:
example = dataset["train"][0]

In [11]:
text = "Call John Smith at 555-1234"
enc = tokenizer(text, return_offsets_mapping=True)
for tok, off in zip(enc.tokens(), enc["offset_mapping"]):
    print(f"{tok:20s} {off}  →  '{text[off[0]:off[1]]}'")

[CLS]                (0, 0)  →  ''
▁Call                (0, 4)  →  'Call'
▁John                (4, 9)  →  ' John'
▁Smith               (9, 15)  →  ' Smith'
▁at                  (15, 18)  →  ' at'
▁555                 (18, 22)  →  ' 555'
-                    (22, 23)  →  '-'
1234                 (23, 27)  →  '1234'
[SEP]                (0, 0)  →  ''


In [ ]:
def tokenize_and_align_labels_single(example):
    """tokenize and align labels for deberta tokenizer.
    since deberta tokenizer will mark space before as part of the next word,
    masks that start the first letter (not white-space) needs this tokenizer.
    it will first get the word_ids that fall in the mask, then label for
    seqeval

    Args:
        example (_type_): _description_

    Returns:
        _type_: _description_
    """
    encoding = tokenizer(
        example["source_text"],
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
        max_length=512
    )
    
    word_ids= encoding.word_ids()
    offsets= encoding["offset_mapping"]
    
    # map offsets to their word_id - for sentence-piece tokenization which 
    # tokenizes the space before the word as part of that word
    word_span: dict[int, tuple[int, int]] = {}
    for offset, word_id in zip(offsets, word_ids):
        if word_id is None:
            continue
        if word_id not in word_span:
            word_span[word_id] = offset
        else:
            prev_start, prev_end = word_span[word_id]
            word_span[word_id] = (
                min(prev_start, offset[0]),
                max(prev_end, offset[1])
            )
    
    # map every word_id to its BI tag
    masks = example["privacy_mask"]
    word_to_ent: dict[int, str] = {}
    for mask in masks:
        words_in_mask = sorted(
            word_id
            for word_id, (w_start, w_end) in word_span.items()
            if w_start < mask["end"] and w_end > mask["start"] 
        )

        for word_id in words_in_mask:
            word_to_ent[word_id] = mask["label"]
    
    # add labels and bio tags
    labels: list[int] = []
    bio_tags: list[Optional[str]] = []
    prev_word_id = None
    prev_ent = None
    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
            bio_tags.append(None)
            
        elif word_id not in word_to_ent:
            labels.append(label2id["O"])
            bio_tags.append("O")
        else: 
            entity = word_to_ent[word_id] 
            
            if word_id != prev_word_id:
                tag = f"I-{entity}" if prev_ent == entity else f"B-{entity}"
                labels.append(label2id[tag])
            else:
                tag = f"I-{entity}"
                labels.append(-100)
                # labels.append(label2id[tag]) # trying non -100
                
            bio_tags.append(tag)
            prev_ent = entity
        prev_word_id = word_id
    
    encoding["ner_tags"] = bio_tags
    encoding["labels"] = labels
    return encoding

In [13]:
dataset = dataset.map(tokenize_and_align_labels_single, batched=False)

Map:   0%|          | 0/29908 [00:00<?, ? examples/s]

Map:   0%|          | 0/3973 [00:00<?, ? examples/s]

Map:   0%|          | 0/3973 [00:00<?, ? examples/s]

In [14]:
example = dataset["train"][0]
example_encodings = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)
line1 = ""
line2 = ""
line3 = ""
line4 = ""
for word, tag, label, word_id in zip(example_encodings.tokens(), example["ner_tags"], example["labels"], example_encodings.word_ids()):
    tag = str(tag)
    word_id = str(word_id)
    label = str(label)
    max_length = max(len(word), len(tag), len(word_id), len(label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += tag + " " * (max_length - len(tag) + 1)
    line3 += label + " " * (max_length - len(label) + 1)
    line4 += word_id + " " * (max_length - len(word_id) + 1)
    
print(line1)
print(line2)
print(line3)
print(line4)

[CLS] ▁Subject : ▁Group ▁Messaging ▁for ▁Admissions ▁Process ▁Good ▁morning , ▁everyone , ▁I ▁hope ▁this ▁message ▁finds ▁you ▁well .  ▁As ▁we ▁continue ▁our ▁admissions ▁processes ,  ▁I ▁would ▁like ▁to ▁update ▁you ▁on ▁the ▁latest ▁developments ▁and ▁key ▁information .  ▁Please ▁find ▁below ▁the ▁timeline ▁for ▁our ▁upcoming ▁meetings :  ▁- ▁          wyn        q          vr         h          053        ▁- ▁Meeting ▁at ▁10    :      20     am     ▁- ▁l         uka        .          burg       ▁- ▁Meeting ▁at ▁21    ▁- ▁qa        hil        .          witt       auer       ▁- ▁Meeting ▁at ▁quarter ▁past  ▁13    ▁- ▁g         hola       m          hos        s          ein        .          ru         schke      ▁- ▁Meeting ▁at ▁9     :      47     ▁PM    ▁- ▁p         d          m          jr         s          y          oz         14         60         [SEP] 
None  O        O O      O          O    O           O        O     O        O O         O O  O     O     O        O      O

In [15]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

seqeval = evaluate.load("seqeval")

In [ ]:

def compute_metrics(p: EvalPrediction) -> dict[str, float]:
    predictions, label_ids = p.predictions, p.label_ids
    predictions = np.argmax(predictions, axis=-1)
    
    true_preds = [
        [id2label[str(pred)] for pred, tgt_id in zip(preds_row, labels_row) if tgt_id != -100]
        for preds_row, labels_row in zip(predictions, label_ids)
    ]
    
    true_labels =[
        [id2label[str(tgt_id)] for tgt_id in labels_row if tgt_id != -100]
        for labels_row in label_ids
    ]
    
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    
    flat = {}
    if results is not None:
        flat.update({
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"],
        })
        
        for entity, scores in results.items():
            if isinstance(scores, dict): # filter scores for individual labels
                flat[f"{entity}_f1"] = scores["f1"]
                flat[f"{entity}_support"] = scores["number"]
    
    return flat

In [17]:
class WeightedTokenClassificationTrainer(Trainer):
    def __init__(self, *args, class_weights: Optional[torch.Tensor]=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        
    def compute_loss(
        self, 
        model: nn.Module, 
        inputs: dict[str, Tensor | Any], 
        return_outputs: bool = False, 
        num_items_in_batch: Tensor | int | None = None
    ) -> Tensor | tuple[Tensor, Any]:
        # get target labels from the inputs
        labels = inputs.get("labels")
        # forward pass
        outputs = model(**inputs)
        
        if labels is None:
            # if no labels during training raise error
            if model.training:
                raise ValueError(
                    "Labels are required during training for WeightedTokenClassificationTrainer."
                )
            # silent fall back for inference (when labels aren't provided)
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss
            return (loss, outputs) if return_outputs else loss
        
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
        
        # get class weights and move to device
        weight = None
        if self.class_weights is not None:
            weight = self.class_weights.to(logits.device)
            
        # def loss function with label weights and ignore -100 in labels    
        loss_fct = nn.CrossEntropyLoss(weight=weight, ignore_index=-100)
        # get loss by comparing predictions to labels
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        
        # for gradient accumulation
        if num_items_in_batch is not None:
            loss = loss/num_items_in_batch
            
        return (loss, outputs) if return_outputs else loss



In [18]:
class DetailedProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and state.is_local_process_zero:
            step = state.global_step
            total = state.max_steps
            loss = logs.get("loss")
            lr = logs.get("learning_rate")
            eval_loss = logs.get("eval_loss")

            if loss is not None and lr is not None:
                # Training step log
                print(f"  Step {step}/{total} | loss: {loss:.4f} | lr: {lr:.2e}")
            elif eval_loss is not None:
                # Eval log
                f1 = logs.get("eval_f1", "—")
                f1_str = f"{f1:.4f}" if isinstance(f1, float) else f1
                print(f"  Eval step {step} | eval_loss: {eval_loss:.4f} | f1: {f1_str}")

In [19]:
for param in model.base_model.parameters():
    param.requires_grad = False
    
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name} requires grad")

classifier.weight requires grad
classifier.bias requires grad


In [20]:
labels_counter = Counter()
for labels in dataset["train"]["labels"]:
    labels_counter.update(labels)

# remove -100 and make tensor
labels_count_tensor = torch.tensor(
    [labels_counter[i] for i in id2label.keys()],
    dtype=torch.float
)
    
weights = 1.0 / labels_count_tensor.clamp(min=1.0)
weights = weights / weights.sum() # normalize

In [21]:
trainer = WeightedTokenClassificationTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[DetailedProgressCallback()],
    class_weights=weights
)
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Bod F1,Bod Support,Building F1,Building Support,City F1,City Support,Country F1,Country Support,Date F1,Date Support,Driverlicense F1,Driverlicense Support,Email F1,Email Support,Geocoord F1,Geocoord Support,Givenname1 F1,Givenname1 Support,Givenname2 F1,Givenname2 Support,Idcard F1,Idcard Support,Ip F1,Ip Support,Lastname1 F1,Lastname1 Support,Lastname2 F1,Lastname2 Support,Lastname3 F1,Lastname3 Support,Pass F1,Pass Support,Passport F1,Passport Support,Postcode F1,Postcode Support,Secaddress F1,Secaddress Support,Sex F1,Sex Support,Socialnumber F1,Socialnumber Support,State F1,State Support,Street F1,Street Support,Tel F1,Tel Support,Time F1,Time Support,Title F1,Title Support,Username F1,Username Support
1,0.000354,0.000159,0.227036,0.371715,0.281896,0.850591,0.261695,1124,0.531705,963,0.436066,989,0.375000,757,0.266955,837,0.095486,1142,0.333429,1206,0.069841,104,0.168848,904,0.033149,255,0.209879,1300,0.370741,1028,0.197721,1158,0.007767,313,0.000000,105,0.440176,784,0.218476,1173,0.390957,954,0.356994,440,0.347548,969,0.158130,1285,0.509468,995,0.430004,967,0.187948,991,0.344607,1825,0.261774,906,0.176883,1295
2,0.000172,0.000147,0.244155,0.408575,0.305657,0.863725,0.304723,1124,0.575291,963,0.477050,989,0.412158,757,0.277802,837,0.135653,1142,0.358693,1206,0.107744,104,0.185637,904,0.035806,255,0.214728,1300,0.390078,1028,0.202026,1158,0.031746,313,0.000000,105,0.464248,784,0.242152,1173,0.389338,954,0.384615,440,0.344384,969,0.167822,1285,0.532293,995,0.465267,967,0.240805,991,0.353204,1825,0.309482,906,0.213439,1295


  Step 500/1870 | loss: 0.0004 | lr: 8.14e-04


/home/zelluzy/Desktop/pii_redaction_api/.venv/lib/python3.14/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


  Eval step 935 | eval_loss: 0.0002 | f1: 0.2819


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Step 1000/1870 | loss: 0.0002 | lr: 5.17e-04
  Step 1500/1870 | loss: 0.0002 | lr: 2.20e-04
  Eval step 1870 | eval_loss: 0.0001 | f1: 0.3057


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1870, training_loss=0.00022363175842213758, metrics={'train_runtime': 94.5002, 'train_samples_per_second': 632.972, 'train_steps_per_second': 19.788, 'total_flos': 3103801989663504.0, 'train_loss': 0.00022363175842213758, 'epoch': 2.0})

In [22]:
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")

In [23]:
# Unfreeze the backbone
for param in model.base_model.parameters():
    param.requires_grad = True

update_args()

trainer = WeightedTokenClassificationTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[DetailedProgressCallback()],
    class_weights=weights
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Bod F1,Bod Support,Building F1,Building Support,City F1,City Support,Country F1,Country Support,Date F1,Date Support,Driverlicense F1,Driverlicense Support,Email F1,Email Support,Geocoord F1,Geocoord Support,Givenname1 F1,Givenname1 Support,Givenname2 F1,Givenname2 Support,Idcard F1,Idcard Support,Ip F1,Ip Support,Lastname1 F1,Lastname1 Support,Lastname2 F1,Lastname2 Support,Lastname3 F1,Lastname3 Support,Pass F1,Pass Support,Passport F1,Passport Support,Postcode F1,Postcode Support,Secaddress F1,Secaddress Support,Sex F1,Sex Support,Socialnumber F1,Socialnumber Support,State F1,State Support,Street F1,Street Support,Tel F1,Tel Support,Time F1,Time Support,Title F1,Title Support,Username F1,Username Support
1,0.000083,0.000032,0.812892,0.894546,0.851766,0.970191,0.941025,1124,0.941949,963,0.936484,989,0.897856,757,0.837472,837,0.814316,1142,0.942261,1206,0.857143,104,0.604707,904,0.495575,255,0.807140,1300,0.954912,1028,0.614366,1158,0.367257,313,0.142349,105,0.941687,784,0.831388,1173,0.935780,954,0.903587,440,0.913306,969,0.854058,1285,0.936734,995,0.905160,967,0.868533,991,0.915994,1825,0.885699,906,0.867122,1295
2,0.000051,0.000029,0.847343,0.923049,0.883577,0.974671,0.872871,1124,0.954268,963,0.940945,989,0.919294,757,0.724098,837,0.860884,1142,0.963083,1206,0.938389,104,0.749758,904,0.625442,255,0.864439,1300,0.968885,1028,0.711877,1158,0.476327,313,0.363636,105,0.939506,784,0.849817,1173,0.954899,954,0.923757,440,0.943683,969,0.896655,1285,0.963555,995,0.923615,967,0.907434,991,0.929778,1825,0.917531,906,0.922673,1295
3,0.000038,0.000026,0.879527,0.934353,0.906112,0.978291,0.955263,1124,0.961637,963,0.945185,989,0.942308,757,0.877412,837,0.887336,1142,0.948811,1206,0.952830,104,0.773109,904,0.630015,255,0.867573,1300,0.983495,1028,0.706633,1158,0.559486,313,0.538860,105,0.944892,784,0.893236,1173,0.963843,954,0.934078,440,0.949873,969,0.921454,1285,0.972250,995,0.940584,967,0.935245,991,0.939669,1825,0.939044,906,0.927252,1295
4,0.000029,0.000023,0.888910,0.943639,0.915457,0.980515,0.958916,1124,0.964213,963,0.952240,989,0.948999,757,0.885284,837,0.900470,1142,0.965629,1206,0.938967,104,0.786653,904,0.701209,255,0.882353,1300,0.964795,1028,0.756296,1158,0.598496,313,0.565737,105,0.954517,784,0.907611,1173,0.965944,954,0.940265,440,0.948705,969,0.920537,1285,0.976143,995,0.947157,967,0.929273,991,0.954582,1825,0.948704,906,0.926920,1295
5,0.000024,0.000023,0.896401,0.938310,0.916877,0.980921,0.950110,1124,0.971780,963,0.956006,989,0.957599,757,0.876968,837,0.892268,1142,0.971126,1206,0.925926,104,0.804071,904,0.696921,255,0.874821,1300,0.988327,1028,0.743787,1158,0.583921,313,0.566802,105,0.957941,784,0.906534,1173,0.962313,954,0.937294,440,0.963608,969,0.927843,1285,0.973619,995,0.947526,967,0.933202,991,0.954779,1825,0.939024,906,0.942123,1295
6,0.000020,0.000025,0.901536,0.945214,0.922859,0.981517,0.963093,1124,0.966650,963,0.958375,989,0.950673,757,0.884507,837,0.910089,1142,0.975112,1206,0.966507,104,0.794347,904,0.698835,255,0.902430,1300,0.984915,1028,0.750000,1158,0.591736,313,0.585586,105,0.958698,784,0.917250,1173,0.969572,954,0.952275,440,0.963228,969,0.931074,1285,0.979612,995,0.956301,967,0.938027,991,0.948988,1825,0.952381,906,0.945315,1295
7,0.000016,0.000026,0.907545,0.947959,0.927312,0.982012,0.966064,1124,0.969231,963,0.966068,989,0.953488,757,0.900693,837,0.920826,1142,0.973856,1206,0.922374,104,0.808383,904,0.726644,255,0.899028,1300,0.974976,1028,0.770192,1158,0.632075,313,0.696078,105,0.956305,784,0.918333,1173,0.964840,954,0.964045,440,0.961793,969,0.928248,1285,0.977556,995,0.958544,967,0.942222,991,0.953175,1825,0.955899,906,0.951464,1295
8,0.000013,0.000025,0.909712,0.945779,0.927395,0.982569,0.955869,1124,0.973168,963,0.965517,989,0.954045,757,0.889144,837,0.918640,1142,0.963355,1206,0.975845,104,0.812339,904,0.744681,255,0.897015,1300,0.972583,1028,0.773978,1158,0.635945,313,0.694064,105,0.956250,784,0.924087,1173,0.965231,954,0.

  Step 500/18700 | loss: 0.0002 | lr: 1.97e-05
  Step 1000/18700 | loss: 0.0001 | lr: 1.91e-05
  Step 1500/18700 | loss: 0.0001 | lr: 1.86e-05
  Eval step 1870 | eval_loss: 0.0000 | f1: 0.8518


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Step 2000/18700 | loss: 0.0001 | lr: 1.80e-05
  Step 2500/18700 | loss: 0.0001 | lr: 1.75e-05
  Step 3000/18700 | loss: 0.0001 | lr: 1.70e-05
  Step 3500/18700 | loss: 0.0001 | lr: 1.64e-05
  Eval step 3740 | eval_loss: 0.0000 | f1: 0.8836


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  Step 15000/18700 | loss: 0.0000 | lr: 4.00e-06
  Step 15500/18700 | loss: 0.0000 | lr: 3.46e-06
  Step 16000/18700 | loss: 0.0000 | lr: 2.92e-06
  Step 16500/18700 | loss: 0.0000 | lr: 2.38e-06
  Eval step 16830 | eval_loss: 0.0000 | f1: 0.9302


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Step 17000/18700 | loss: 0.0000 | lr: 1.84e-06
  Step 17500/18700 | loss: 0.0000 | lr: 1.30e-06
  Step 18000/18700 | loss: 0.0000 | lr: 7.57e-07
  Step 18500/18700 | loss: 0.0000 | lr: 2.17e-07
  Eval step 18700 | eval_loss: 0.0000 | f1: 0.9299


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18700, training_loss=3.278237401462972e-05, metrics={'train_runtime': 1069.8512, 'train_samples_per_second': 279.553, 'train_steps_per_second': 17.479, 'total_flos': 1.4289844674308136e+16, 'train_loss': 3.278237401462972e-05, 'epoch': 10.0})

In [24]:
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")
        

In [25]:
example = dataset["train"][0]
example_encodings = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)
line1 = ""
line2 = ""
line3 = ""
line4 = ""
for word, tag, label, word_id in zip(example_encodings.tokens(), example["ner_tags"], example["labels"], example_encodings.word_ids()):
    tag = str(tag)
    word_id = str(word_id)
    label = str(label)
    max_length = max(len(word), len(tag), len(word_id), len(label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += tag + " " * (max_length - len(tag) + 1)
    line3 += label + " " * (max_length - len(label) + 1)
    line4 += word_id + " " * (max_length - len(word_id) + 1)
    
print(line1)
print(line2)
print(line3)
print(line4)

[CLS] ▁Subject : ▁Group ▁Messaging ▁for ▁Admissions ▁Process ▁Good ▁morning , ▁everyone , ▁I ▁hope ▁this ▁message ▁finds ▁you ▁well .  ▁As ▁we ▁continue ▁our ▁admissions ▁processes ,  ▁I ▁would ▁like ▁to ▁update ▁you ▁on ▁the ▁latest ▁developments ▁and ▁key ▁information .  ▁Please ▁find ▁below ▁the ▁timeline ▁for ▁our ▁upcoming ▁meetings :  ▁- ▁          wyn        q          vr         h          053        ▁- ▁Meeting ▁at ▁10    :      20     am     ▁- ▁l         uka        .          burg       ▁- ▁Meeting ▁at ▁21    ▁- ▁qa        hil        .          witt       auer       ▁- ▁Meeting ▁at ▁quarter ▁past  ▁13    ▁- ▁g         hola       m          hos        s          ein        .          ru         schke      ▁- ▁Meeting ▁at ▁9     :      47     ▁PM    ▁- ▁p         d          m          jr         s          y          oz         14         60         [SEP] 
None  O        O O      O          O    O           O        O     O        O O         O O  O     O     O        O      O

In [26]:
Counter([param.dtype for param in model.parameters()])

Counter({torch.float32: 104})

In [27]:
import html as html_lib

def log_pii_test_results(
    trainer,
    test_dataset,
    tokenizer,
    table_name: str = "pii_test_results",
    run: Optional[wandb.sdk.wandb_run.Run] = None,
) -> wandb.Table:
    # Bug 1 fix: resolve to a concrete run object, not the wandb module.
    # wandb.run is the currently active run (set by wandb.init or the Trainer callback).
    active_run = run or wandb.run
    if active_run is None:
        raise RuntimeError(
            "No active W&B run found. Either pass `run=` explicitly "
            "or call wandb.init() before invoking this function."
        )

    # Bug 2 fix: normalize to int keys — live models use int, JSON-reloaded models use str.
    id2label = {int(k): v for k, v in trainer.model.config.id2label.items()}

    pred_output = trainer.predict(test_dataset)
    pred_ids = np.argmax(pred_output.predictions, axis=-1)
    true_ids = pred_output.label_ids

    table = wandb.Table(columns=["id", "annotated_sequence", "f1", "precision", "recall", "tp", "fp", "fn"])

    for i in range(len(test_dataset)):
        tokens, true_labels, pred_labels = [], [], []
        for token_id, true_id, pred_id in zip(test_dataset[i]["input_ids"], true_ids[i], pred_ids[i]):
            if true_id == -100:
                continue
            tokens.append(tokenizer.convert_ids_to_tokens(int(token_id)))
            true_labels.append(id2label[int(true_id)])
            pred_labels.append(id2label[int(pred_id)])

        tp = fp = fn = 0
        for true, pred in zip(true_labels, pred_labels):
            if true != "O" and pred == true:   tp += 1
            elif true != "O" and pred != true: fn += 1
            elif true == "O" and pred != "O":  fp += 1

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        parts = []
        for token, true, pred in zip(tokens, true_labels, pred_labels):
            t = html_lib.escape(token)
            if true != "O" and pred == true:
                badge = f'<sup style="font-size:9px;padding:1px 4px;border-radius:3px;background:#C0DD97;color:#27500A">{html_lib.escape(true)}</sup>'
                parts.append(f'<span style="background:#C0DD97;color:#27500A;font-weight:500;padding:2px 4px;border-radius:3px;margin:1px 2px;display:inline-block">{t}{badge}</span>')
            elif true != "O" and pred != true:
                label = html_lib.escape(f"{true}→{pred}")
                badge = f'<sup style="font-size:9px;padding:1px 4px;border-radius:3px;background:#F7C1C1;color:#791F1F">{label}</sup>'
                parts.append(f'<span style="background:#F7C1C1;color:#791F1F;font-weight:500;padding:2px 4px;border-radius:3px;margin:1px 2px;display:inline-block">{t}{badge}</span>')
            elif true == "O" and pred != "O":
                badge = f'<sup style="font-size:9px;padding:1px 4px;border-radius:3px;background:#FAC775;color:#633806">{html_lib.escape(pred)}</sup>'
                parts.append(f'<span style="background:#FAC775;color:#633806;font-weight:500;padding:2px 4px;border-radius:3px;margin:1px 2px;display:inline-block">{t}{badge}</span>')
            else:
                parts.append(f'<span style="margin:1px 2px;display:inline-block">{t}</span>')

        seq_html = '<div style="font-family:monospace;font-size:12px;line-height:2">' + "".join(parts) + "</div>"
        table.add_data(i, wandb.Html(seq_html), round(f1, 4), round(precision, 4), round(recall, 4), tp, fp, fn)

    active_run.log({table_name: table})
    return table

In [28]:
table = log_pii_test_results(trainer, dataset["test"], tokenizer, run=run)

In [29]:
# def log_predictions_table(trainer: Trainer, test_ds: Dataset | DatasetDict, 
#                           tokenizer: BertTokenizer, id2label: dict, 
#                           n_samples:int=50) -> wandb.Table:
#     predictions, labels, _ = trainer.predict(test_ds) # type:ignore
#     preds = np.argmax(predictions, axis=2)
    
#     table = wandb.Table(columns=["sequence", "true_labels", "pred_labels", "accuracy"])
    
#     for i in range(min(len(test_ds), n_samples)):
#         tokens = tokenizer.convert_ids_to_tokens(test_ds[i]["input_ids"])
        
#         row_tokens, true_row, pred_row = [], [], []
#         for token, true_id, pred_id in zip(tokens, labels[i], preds[i]):
#             true_id = str(true_id)
#             pred_id = str(pred_id)
#             if true_id == "-100":
#                 continue
#             row_tokens.append(token)
#             true_row.append(id2label[true_id])
#             pred_row.append(id2label[pred_id])

#         correct = np.where(np.array(true_row)==np.array(pred_row), 1, 0) 
#         accuracy = float(np.sum(correct) / len(correct)) * 100
#         table.add_data(
#             " ".join(row_tokens),
#             " ".join(true_row),
#             " ".join(pred_row),
#             f"{accuracy:.2f}%"
#         )
        
#     wandb.log({"predictions": table})
#     return table
            

In [30]:
# table2 = log_predictions_table(trainer=trainer, test_ds=dataset["test"], tokenizer=tokenizer, id2label=id2label)

In [29]:
pred_output = trainer.predict(
    test_dataset=dataset["test"]
)
f1s = []
for metric, value in pred_output.metrics.items():
    if metric.endswith("f1"):
        f1s.append((metric, value))
        
sorted_f1s = sorted(f1s, key=lambda x: x[1], reverse=True)

print("worst f1s:")
for metric, value in reversed(sorted_f1s[-7:]):
    print(f"{metric:<20}= {value:.3f}")

worst f1s:
test_LASTNAME3_f1   = 0.663
test_LASTNAME2_f1   = 0.697
test_GIVENNAME2_f1  = 0.741
test_LASTNAME1_f1   = 0.790
test_GIVENNAME1_f1  = 0.826
test_DATE_f1        = 0.887
test_IDCARD_f1      = 0.889


In [30]:
wandb.finish()

eval/BOD_f1,▁▁█▇████████
eval/BOD_support,▁▁▁▁▁▁▁▁▁▁▁▁
eval/BUILDING_f1,▁▂██████████
eval/BUILDING_support,▁▁▁▁▁▁▁▁▁▁▁▁
eval/CITY_f1,▁▂██████████
eval/CITY_support,▁▁▁▁▁▁▁▁▁▁▁▁
eval/COUNTRY_f1,▁▁▇█████████
eval/COUNTRY_support,▁▁▁▁▁▁▁▁▁▁▁▁
eval/DATE_f1,▁▁▇▆████████
eval/DATE_support,▁▁▁▁▁▁▁▁▁▁▁▁
+119,...
